&nbsp;
# Does Batch Normalisation Make Training Less Fragile?

**Question:** how much does the learning rate matter for a small CNN on Fashion-MNIST, and does batch norm widen the range of learning rates that work?

Built on the class notebook `hyperparameters_lightning.ipynb`.

**Design (one thing changes at a time):**
- Manipulated: `lr` in 5 values (log scale), measured with batch norm ON and OFF.
- Fixed: Adam, batch size 64, dropout 0.25, 10,000 training images, 5 epochs, no early stopping, no augmentation, same architecture.
- Seeds 42, 43, 44 so we can tell real differences from noise.
- Everything is judged on the **validation** set. The test set is never touched.

 "If I pick a bad learning rate, does batch norm save me?" Say what you did (a small CNN on Fashion-MNIST, 5 learning rates, with and without batch norm) and tease the answer without giving every number.*

## What is the learning rate, and what does batch norm do?

- *Update rule θ ← θ − η∇L. What too small / too large look like.*
- *Batch norm: normalise each channel with the batch mean and variance, then rescale with learned γ and β. Include the formula:*
  x̂ = (x − μ_B) / √(σ²_B + ε),  y = γ x̂ + β
- *Why it might make training less sensitive to the learning rate. Check your sources for the current explanation, because the original "internal covariate shift" story has been challenged. Say which view you found convincing and cite it.*
- *A diagram of where the BN layer sits in the conv block (Conv → BN → activation → pool) would help.*

## Experiment setup

| Item | Value |
|---|---|
| Dataset | Fashion-MNIST, 10,000-image training subset, 10% of the full train set held out for validation |
| Model | 2 conv blocks + 2 linear layers, N parameters (BN on / off) |
| Optimizer | Adam, batch size 64, dropout 0.25 |
| Epochs | 5, no early stopping |
| **Varied** | learning rate ∈ {1e-4, 1e-3, 1e-2, 3e-2, 1e-1}; batch norm on / off |
| Seeds | 42, 43, 44 |

In [ ]:
# %pip install torch torchvision lightning pandas matplotlib

In [ ]:
import logging
import random
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Lightning 2.x is imported as `lightning`; older installs use `pytorch_lightning`.
try:
    import lightning as L
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
    _LIGHTNING_LOGGER = "lightning.pytorch"
except ImportError:
    import pytorch_lightning as L
    from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
    from pytorch_lightning.loggers import CSVLogger
    _LIGHTNING_LOGGER = "pytorch_lightning"

# Keep the output readable during the many short training runs below.
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=".*logging interval.*")

print("torch:", torch.__version__)
print("lightning:", L.__version__)
print("GPU available:", torch.cuda.is_available())

## 1. Framework from the class notebook

In [ ]:
@dataclass
class HParams:
    # ---- Data -------------------------------------------------------------
    data_dir: str = "./data"
    batch_size: int = 64          # samples per gradient step
    val_fraction: float = 0.1     # share of the training set held out for validation
    train_subset: Optional[int] = None  # use only N training images (None = all); for fast experiments
    augment: bool = False         # random crop + horizontal flip on training images
    num_workers: int = 0          # data-loading processes (not a model hyperparameter; 0 is safest in notebooks)

    # ---- Model ------------------------------------------------------------
    conv1_channels: int = 32      # width of the first conv block
    conv2_channels: int = 64      # width of the second conv block
    hidden_dim: int = 128         # size of the fully connected layer
    activation: str = "relu"      # relu | leaky_relu | gelu | tanh
    use_batchnorm: bool = True    # batch normalisation after each conv
    dropout: float = 0.25         # probability of zeroing an activation in the classifier head

    # ---- Optimisation -----------------------------------------------------
    optimizer: str = "adam"       # sgd | adam | adamw
    lr: float = 1e-3              # learning rate (step size)
    momentum: float = 0.9         # used by SGD only
    weight_decay: float = 0.0     # L2-style penalty on the weights
    scheduler: str = "none"       # none | step | cosine

    # ---- Training loop ----------------------------------------------------
    max_epochs: int = 5           # upper limit on passes over the training set
    patience: int = 3             # early stopping: epochs without val_loss improvement before stopping
    seed: int = 42                # controls weight init, data split and shuffling


hp = HParams()
pd.Series(asdict(hp), name="value").to_frame()

In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Mean and standard deviation of the Fashion-MNIST training pixels.
FMNIST_MEAN, FMNIST_STD = (0.2860,), (0.3530,)


class FashionMNISTDataModule(L.LightningDataModule):
    def __init__(self, hp: HParams):
        super().__init__()
        self.hp = hp
        normalize = transforms.Normalize(FMNIST_MEAN, FMNIST_STD)
        self.eval_transform = transforms.Compose([transforms.ToTensor(), normalize])

        augmentations = []
        if hp.augment:  # <-- hyperparameter: augment
            augmentations = [transforms.RandomCrop(28, padding=2),
                             transforms.RandomHorizontalFlip()]
        self.train_transform = transforms.Compose([*augmentations, transforms.ToTensor(), normalize])

    def prepare_data(self):
        # Download once. Lightning calls this on a single process.
        datasets.FashionMNIST(self.hp.data_dir, train=True, download=True)
        datasets.FashionMNIST(self.hp.data_dir, train=False, download=True)

    def setup(self, stage=None):
        # Two views of the same 60k images: one with training transforms, one without.
        train_view = datasets.FashionMNIST(self.hp.data_dir, train=True, transform=self.train_transform)
        eval_view = datasets.FashionMNIST(self.hp.data_dir, train=True, transform=self.eval_transform)

        n_total = len(train_view)
        n_val = int(n_total * self.hp.val_fraction)  # <-- hyperparameter: val_fraction
        generator = torch.Generator().manual_seed(self.hp.seed)
        indices = torch.randperm(n_total, generator=generator).tolist()
        val_idx, train_idx = indices[:n_val], indices[n_val:]

        if self.hp.train_subset is not None:  # <-- hyperparameter: train_subset
            train_idx = train_idx[: self.hp.train_subset]

        self.train_ds = Subset(train_view, train_idx)
        self.val_ds = Subset(eval_view, val_idx)
        self.test_ds = datasets.FashionMNIST(self.hp.data_dir, train=False, transform=self.eval_transform)

    def _loader(self, dataset, batch_size, shuffle):
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          num_workers=self.hp.num_workers,
                          persistent_workers=self.hp.num_workers > 0)

    def train_dataloader(self):
        return self._loader(self.train_ds, self.hp.batch_size, shuffle=True)  # <-- hyperparameter: batch_size

    def val_dataloader(self):
        # The evaluation batch size only affects speed, never the result.
        return self._loader(self.val_ds, 512, shuffle=False)

    def test_dataloader(self):
        return self._loader(self.test_ds, 512, shuffle=False)

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU, "gelu": nn.GELU, "tanh": nn.Tanh}


class SmallCNN(nn.Module):
    def __init__(self, hp: HParams, num_classes: int = 10):
        super().__init__()
        act = ACTIVATIONS[hp.activation]            # <-- hyperparameter: activation
        c1, c2 = hp.conv1_channels, hp.conv2_channels  # <-- hyperparameters: width

        def conv_block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
            if hp.use_batchnorm:                    # <-- hyperparameter: use_batchnorm
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [act(), nn.MaxPool2d(2)]
            return nn.Sequential(*layers)

        self.features = nn.Sequential(conv_block(1, c1), conv_block(c1, c2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(hp.dropout),                 # <-- hyperparameter: dropout
            nn.Linear(c2 * 7 * 7, hp.hidden_dim),   # <-- hyperparameter: hidden_dim
            act(),
            nn.Dropout(hp.dropout),
            nn.Linear(hp.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = SmallCNN(hp)
print(model)
print(f"\ntrainable parameters: {count_parameters(model):,}")

In [ ]:
class LitClassifier(L.LightningModule):
    def __init__(self, hp: HParams):
        super().__init__()
        self.hp = hp
        self.save_hyperparameters(asdict(hp))  # logged + stored in checkpoints
        self.model = SmallCNN(hp)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage: str):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        # on_epoch=True: Lightning averages over the epoch (weighted by batch size).
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        hp = self.hp
        params = self.parameters()

        # <-- hyperparameters: optimizer, lr, momentum, weight_decay
        if hp.optimizer == "sgd":
            optimizer = torch.optim.SGD(params, lr=hp.lr, momentum=hp.momentum, weight_decay=hp.weight_decay)
        elif hp.optimizer == "adam":
            optimizer = torch.optim.Adam(params, lr=hp.lr, weight_decay=hp.weight_decay)
        elif hp.optimizer == "adamw":
            optimizer = torch.optim.AdamW(params, lr=hp.lr, weight_decay=hp.weight_decay)
        else:
            raise ValueError(f"unknown optimizer: {hp.optimizer}")

        # <-- hyperparameter: scheduler
        if hp.scheduler == "none":
            return optimizer
        if hp.scheduler == "step":
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, hp.max_epochs // 3), gamma=0.1)
        elif hp.scheduler == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp.max_epochs)
        else:
            raise ValueError(f"unknown scheduler: {hp.scheduler}")
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

In [ ]:
def load_history(log_dir) -> pd.DataFrame:
    """Read Lightning's metrics.csv and return one row per epoch."""
    df = pd.read_csv(Path(log_dir) / "metrics.csv")
    cols = [c for c in ["train_loss", "val_loss", "train_acc", "val_acc"] if c in df.columns]
    return df.groupby("epoch")[cols].mean().reset_index()


def fit(hp: HParams, run_name: str = "run", verbose: bool = True, checkpoint: bool = False):
    """Train one model from a config. Returns (trainer, model, datamodule, history)."""
    for name in (_LIGHTNING_LOGGER, "lightning.fabric", "lightning_fabric"):
        logging.getLogger(name).setLevel(logging.INFO if verbose else logging.ERROR)
    L.seed_everything(hp.seed, workers=True)  # <-- hyperparameter: seed

    datamodule = FashionMNISTDataModule(hp)
    model = LitClassifier(hp)
    logger = CSVLogger("logs", name=run_name)

    callbacks = [EarlyStopping(monitor="val_loss", mode="min", patience=hp.patience)]  # <-- hyperparameter: patience
    if checkpoint:
        callbacks.append(ModelCheckpoint(monitor="val_acc", mode="max", save_top_k=1))

    trainer = L.Trainer(
        max_epochs=hp.max_epochs,          # <-- hyperparameter: max_epochs
        accelerator="auto",                # GPU if there is one, otherwise CPU
        devices=1,
        logger=logger,
        callbacks=callbacks,
        enable_checkpointing=checkpoint,
        enable_progress_bar=verbose,
        enable_model_summary=verbose,
        log_every_n_steps=10,
    )
    trainer.fit(model, datamodule=datamodule)
    history = load_history(logger.log_dir)
    return trainer, model, datamodule, history

## 2. Experiment configuration

In [ ]:
BASE = replace(HParams(), optimizer="adam", batch_size=64, dropout=0.25,
               train_subset=10_000, max_epochs=5, patience=100, augment=False)

LRS   = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1]
BN    = [True, False]
SEEDS = [42, 43, 44]

setup = pd.Series({k: getattr(BASE, k) for k in
    ["optimizer","batch_size","dropout","train_subset","max_epochs","val_fraction","augment"]})
setup["seeds"] = SEEDS
setup["lr values"] = LRS
setup["trainable params (BN on)"]  = count_parameters(SmallCNN(replace(BASE, use_batchnorm=True)))
setup["trainable params (BN off)"] = count_parameters(SmallCNN(replace(BASE, use_batchnorm=False)))
setup.to_frame("value")

## 3. Run everything
About 30 short runs. If it is too slow on your machine, reduce `SEEDS` to `[42]` first.

In [ ]:
runs = {}   # (use_bn, lr, seed) -> history
for bn in BN:
    for lr in LRS:
        for seed in SEEDS:
            hp_i = replace(BASE, use_batchnorm=bn, lr=lr, seed=seed)
            _, _, _, h = fit(hp_i, run_name=f"bn_{bn}/lr_{lr:g}/seed_{seed}", verbose=False)
            runs[(bn, lr, seed)] = h
            print(f"BN={bn!s:<5} lr={lr:<7g} seed={seed}  best val_acc={h['val_acc'].max():.4f}")

## 4. Figure 1: learning curves (seed 42)
Solid = validation, dashed = training. One panel per setting of batch norm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
colors = plt.cm.viridis([i / (len(LRS) - 1) for i in range(len(LRS))])
for ax, bn in zip(axes, BN):
    for c, lr in zip(colors, LRS):
        h = runs[(bn, lr, 42)]
        ax.plot(h["epoch"], h["val_loss"],   "-",  color=c, marker="o", label=f"lr={lr:g}")
        ax.plot(h["epoch"], h["train_loss"], "--", color=c)
    ax.set(title=f"Batch norm {'ON' if bn else 'OFF'}", xlabel="epoch", ylim=(0, 3))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Figure 2: final validation accuracy vs learning rate (mean ± std over seeds)

In [ ]:
rows = []
for (bn, lr, seed), h in runs.items():
    rows.append(dict(batchnorm=bn, lr=lr, seed=seed,
                     final_val_acc=h["val_acc"].iloc[-1], best_val_acc=h["val_acc"].max(),
                     final_val_loss=h["val_loss"].iloc[-1], final_train_loss=h["train_loss"].iloc[-1]))
df = pd.DataFrame(rows)
agg = df.groupby(["batchnorm", "lr"]).agg(
    val_acc_mean=("final_val_acc", "mean"), val_acc_std=("final_val_acc", "std"),
    val_loss_mean=("final_val_loss", "mean")).reset_index()

plt.figure(figsize=(6, 4))
for bn in BN:
    a = agg[agg.batchnorm == bn]
    plt.errorbar(a["lr"], a["val_acc_mean"], yerr=a["val_acc_std"], marker="o", capsize=3,
                 label=f"batch norm {'on' if bn else 'off'}")
plt.xscale("log"); plt.xlabel("learning rate (log scale)"); plt.ylabel("final validation accuracy")
plt.axhline(0.1, color="gray", ls=":", label="chance")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## 6. Table: results by setting

In [ ]:
table = agg.pivot(index="lr", columns="batchnorm", values=["val_acc_mean", "val_acc_std"]).round(4)
table

## Discussion
*Answer these in order:*
1. *What happened? Which lr values worked with and without BN?*
2. *Why do you think so? Connect it to the theory in your explanation section.*
3. *Did it match what you expected? Be honest if it didn't.*
4. *Which differences are bigger than the seed-to-seed spread, and which are not?*
5. *At least one limitation (subset, 5 epochs, only Adam, 3 seeds, small model, one dataset).*
